# 14 — Golden Reference for Single-Concept Top-k CGD

Freezes the behavior of the original `generate_with_topk_guide` before any refactoring.
Runs 10 fixed prompts against 3 fixed guides (k=3, steps=16) on the AWQ Llama 3.1 8B and
saves texts + final probe values to `tests/golden/single_concept.json`.

`tests/test_golden.py` regression-tests every later change against this file.

Determinism notes: decoding is greedy (no sampling, no seed), but the golden file is only
valid for the same batch composition (all 10 prompts in one batch) and the same GPU/driver.
Model is pinned to a single device (`cuda:0`) to avoid sharding nondeterminism.

In [ ]:
import json
import platform
from pathlib import Path

import torch
import transformers

from frames.representations import FrameUnembeddingRepresentation

In [ ]:
MODEL_ID = "hugging-quants/Meta-Llama-3.1-8B-Instruct-AWQ-INT4"
DEVICE_MAP = "cuda:0"
K = 3
STEPS = 16
MIN_LEMMAS_PER_SYNSET = 11
MAX_TOKEN_COUNT = 3
GOLDEN_PATH = Path("tests/golden/single_concept.json")

GUIDES = [
    ["dog.n.01"],
    ["joy.n.01"],
    ["woman.n.01", "man.n.01"],
]

In [ ]:
def chat(user: str, assistant: str = "") -> str:
    return (
        f"<|start_header_id|>user<|end_header_id|>{user}<|eot_id|>"
        f"<|start_header_id|>assistant<|end_header_id|>{assistant}"
    )


PROMPTS = [
    chat("What men can be?", "1."),
    chat("What women can be?", "1."),
    chat("Tell me a short story.", "Once"),
    chat("Describe your favorite place.", "My favorite place is"),
    chat("What do you like to do on weekends?", "I like"),
    chat("Give me some advice for a good life.", "1."),
    chat("What is the most important thing in the world?", "The most important thing is"),
    chat("Describe the weather today.", "Today the weather is"),
    chat("What should I cook for dinner?", "You should"),
    chat("Name things that make people happy.", "1."),
]

len(PROMPTS)

In [ ]:
fur = FrameUnembeddingRepresentation.from_model_id(
    MODEL_ID,
    device_map=DEVICE_MAP,
    torch_dtype=torch.float16,
)

Early check that every guide synset survives the `min_lemmas_per_synset` /
`max_token_count` filter — fail loudly here rather than mid-generation.

In [ ]:
for guide in GUIDES:
    for synset in guide:
        concept = fur.get_concept(synset, MIN_LEMMAS_PER_SYNSET, MAX_TOKEN_COUNT)
        print(synset, "->", concept)

In [ ]:
runs = []
for guide in GUIDES:
    texts, probe = fur.quick_generate_with_topk_guide(
        PROMPTS,
        guide=guide,
        min_lemmas_per_synset=MIN_LEMMAS_PER_SYNSET,
        max_token_count=MAX_TOKEN_COUNT,
        k=K,
        steps=STEPS,
    )
    runs.append(
        {
            "guide": guide,
            "texts": texts,
            "probe_final": probe[..., -1].tolist(),
        }
    )
    print(f"=== guide={guide}")
    for text in texts:
        print("   ", text.replace("\n", " ")[-120:])

In [ ]:
GOLDEN_PATH.parent.mkdir(parents=True, exist_ok=True)

golden = {
    "config": {
        "model_id": MODEL_ID,
        "device_map": DEVICE_MAP,
        "torch_dtype": "float16",
        "k": K,
        "steps": STEPS,
        "min_lemmas_per_synset": MIN_LEMMAS_PER_SYNSET,
        "max_token_count": MAX_TOKEN_COUNT,
    },
    "environment": {
        "python": platform.python_version(),
        "torch": torch.__version__,
        "transformers": transformers.__version__,
        "gpu": torch.cuda.get_device_name(0),
    },
    "prompts": PROMPTS,
    "runs": runs,
}

GOLDEN_PATH.write_text(json.dumps(golden, indent=2, ensure_ascii=False))
print(f"saved {GOLDEN_PATH.resolve()}")